# 01 · Data preparation

Loads the raw files from `data/raw/`, cleans them and builds four analysis tables in `data/processed/`.

| Table | One row = | Used for |
|---|---|---|
| `incidents.csv` | one TTC bus incident | route / location analysis |
| `events.csv` | one event on one day | event calendar |
| `daily.csv` | one day | main event-effect analysis |
| `hourly.csv` | one hour | effect in the hours around events |

All dates and times are **Toronto local time**.

## 1. Setup
Paths, analysis period and the event time window rule.

In [1]:
import json
from pathlib import Path

import holidays
import pandas as pd

RAW = Path("../data/raw")
PROCESSED = Path("../data/processed")

# Analysis period = full coverage of the TTC file
START, END = pd.Timestamp("2025-01-01"), pd.Timestamp("2026-09-30")
TZ = "America/Toronto"

# Event time window: from 2 h before the start to 2 h after the end.
# If the end time is unknown, the event is assumed to last 3 h.
HOURS_BEFORE, HOURS_AFTER, DEFAULT_DURATION = 2, 2, 3

# Delay causes most related to events (see Code Descriptions.csv)
KEY_CODES = {"MFDV": "n_diversion", "MFPR": "n_parade_hold", "MFUS": "n_behind_schedule"}

## 2. TTC bus delays

### 2.1 Load
One row = one incident logged by TTC staff. `Min Delay` = delay in minutes (not a minimum).

In [2]:
ttc_raw = pd.read_csv(RAW / "TTC Bus Delay Data since 2025.csv")
print(ttc_raw.shape)
ttc_raw.head()

(107349, 11)


,_id,Date,Line,Time,Day,Station,Code,Min Delay,Min Gap,Bound,Vehicle
0,1,2025-01-01T00:00:00,102 MARKHAM ROAD,02:15,Wednesday,WARDEN STATION,MFESA,20,40,N,3442
1,2,2025-01-01T00:00:00,65 PARLIAMENT,02:15,Wednesday,KIPLING STATION,MFUS,0,0,NaN,0
2,3,2025-01-01T00:00:00,64 MAIN,02:40,Wednesday,BROADVIEW STATION,MFUI,0,0,NaN,8546
3,4,2025-01-01T00:00:00,100 FLEMINGDON PARK,02:43,Wednesday,OVERLEA AND THORNCLIFF,MFSAN,17,32,N,8693
4,5,2025-01-01T00:00:00,34 EGLINTON EAST,03:05,Wednesday,EGLINTON AND DON MILLS,MFUI,20,40,W,8801


### 2.2 Remove duplicates
Rows identical in every column (except the row id) are double entries.

In [3]:
is_duplicate = ttc_raw.drop(columns="_id").duplicated()
print("Exact duplicate rows removed:", is_duplicate.sum())
ttc_unique = ttc_raw[~is_duplicate]

Exact duplicate rows removed: 300


### 2.3 Rename columns, build date-time
- Short `snake_case` names; `Bound` and `Vehicle` are not used.
- `Day` is dropped: on 2026-04-26 it says "Saturday" for a Sunday. The weekday is taken from `Date` later.

In [ ]:
date = pd.to_datetime(ttc_unique["Date"])
wrong_day = date.dt.day_name() != ttc_unique["Day"]
print("Rows where Day does not match Date:", wrong_day.sum(), "| dates:", date[wrong_day].dt.date.unique())

ttc = (
    ttc_unique.rename(columns={"Line": "line", "Station": "location", "Code": "code",
                               "Min Delay": "min_delay", "Min Gap": "min_gap"})
    .assign(date=date, datetime=lambda d: date + pd.to_timedelta(d["Time"] + ":00"))
    .assign(hour=lambda d: d["datetime"].dt.hour)
    [["date", "datetime", "hour", "line", "location", "code", "min_delay", "min_gap"]]
)

ttc.head()

Rows where Day does not match Date: 130 | dates: [datetime.date(2026, 4, 26)]


,date,datetime,hour,line,location,code,min_delay,min_gap
0,2025-01-01,2025-01-01 02:15:00,2,102 MARKHAM ROAD,WARDEN STATION,MFESA,20,40
1,2025-01-01,2025-01-01 02:15:00,2,65 PARLIAMENT,KIPLING STATION,MFUS,0,0
2,2025-01-01,2025-01-01 02:40:00,2,64 MAIN,BROADVIEW STATION,MFUI,0,0
3,2025-01-01,2025-01-01 02:43:00,2,100 FLEMINGDON PARK,OVERLEA AND THORNCLIFF,MFSAN,17,32
4,2025-01-01,2025-01-01 03:05:00,3,34 EGLINTON EAST,EGLINTON AND DON MILLS,MFUI,20,40


### 2.4 Route number
`line` is free text ("29 DUFFERIN", "29 DUFFERIN EXPRESS"). We take the leading route number.
Shuttles, training buses and empty values get no number.

In [5]:
ttc["route"] = pd.to_numeric(ttc["line"].str.extract(r"^(\d{1,3})\b")[0]).astype("Int64")
print("Rows without route number:", ttc["route"].isna().sum())
ttc.loc[ttc["route"].isna(), "line"].value_counts(dropna=False).head(8)

Rows without route number: 1826


line
NaN               1314
LINE 1 SHUTTLE      73
LINE 2 SHUTTLE      52
YU                  30
SRT                 28
RAD                 25
SHUTTLE BUS         23
BD                  21
Name: count, dtype: int64

### 2.5 Cause codes
- Add the description from `Code Descriptions.csv` (and fix a broken dash character in it).
- Codes first used in 2026 are missing from that table → "Not in reference table".
- `cause_group` = first letter of the code; group names follow the code descriptions.

In [6]:
codes = (
    pd.read_csv(RAW / "Code Descriptions.csv")
    .rename(columns={"CODE": "code", "DESCRIPTION": "code_description"})
    .assign(code_description=lambda d: d["code_description"].str.replace("\u00e2\u0080\u0093", "-"))
)
ttc = ttc.merge(codes[["code", "code_description"]], on="code", how="left", validate="many_to_one")

unknown_code = ttc["code_description"].isna()
print("Rows with a code not in the reference table:", unknown_code.sum(),
      "| years:", ttc.loc[unknown_code, "date"].dt.year.unique())
ttc["code_description"] = ttc["code_description"].fillna("Not in reference table")

CAUSE_GROUPS = {"E": "Equipment", "M": "Miscellaneous", "P": "Other", "S": "Security", "T": "Operator"}
ttc["cause_group"] = ttc["code"].str[0].map(CAUSE_GROUPS).fillna("Unknown")
ttc["cause_group"].value_counts()

Rows with a code not in the reference table: 6913 | years: [2026]


cause_group
Miscellaneous    38646
Equipment        34800
Operator         25420
Security          8122
Other               60
Unknown              1
Name: count, dtype: int64

### 2.6 Zero and extreme delays
- `min_delay = 0`: incident logged, but no service delay. Rows are kept and flagged with `is_delay`.
- A few delays are extreme (up to 999 min, mostly long diversions; 997–999 look like placeholders).
  `delay_capped` caps them at the 99th percentile so they do not dominate totals.

In [7]:
ttc["is_delay"] = ttc["min_delay"] > 0
cap = ttc.loc[ttc["is_delay"], "min_delay"].quantile(0.99, interpolation="lower")
ttc["delay_capped"] = ttc["min_delay"].clip(upper=cap)

print(f"Zero-delay rows: {(~ttc['is_delay']).sum():,}")
print(f"Cap (99th percentile of positive delays): {cap:.0f} min | rows capped: {(ttc['min_delay'] > cap).sum():,}")

Zero-delay rows: 13,774
Cap (99th percentile of positive delays): 294 min | rows capped: 932


## 3. Event calendar
All events get one format: `event_id, event_type, event_name, venue, first_day, last_day, start, end`.

### 3.1 Sports home games
- Each file has home and away games; we keep Toronto home games.
- API times are UTC → converted to Toronto time **before** taking the date
  (World Series Game 7: 2025-11-02 UTC = evening of 2025-11-01 in Toronto).

In [8]:
def read_json(file):
    return json.loads((RAW / file).read_text(encoding="utf-8"))


def mlb_home_games(file):
    games = [g for day in read_json(file)["dates"] for g in day["games"]]
    return [{"event_id": f"mlb_{g['gamePk']}", "event_type": "mlb",
             "event_name": f"Blue Jays vs {g['teams']['away']['team']['name']}",
             "venue": g["venue"]["name"], "start_utc": g["gameDate"], "status": g["status"]["detailedState"]}
            for g in games if g["teams"]["home"]["team"]["name"] == "Toronto Blue Jays"]


def nhl_home_games(file):
    return [{"event_id": f"nhl_{g['id']}", "event_type": "nhl",
             "event_name": f"Maple Leafs vs {g['awayTeam']['abbrev']}",
             "venue": g["venue"]["default"], "start_utc": g["startTimeUTC"], "status": g["gameState"]}
            for g in read_json(file)["games"] if g["homeTeam"]["abbrev"] == "TOR"]


def espn_home_games(file, event_type, team):
    games = []
    for e in read_json(file)["events"]:
        match = e["competitions"][0]
        teams = {c["homeAway"]: c["team"]["displayName"] for c in match["competitors"]}
        if teams["home"] == team:
            games.append({"event_id": f"{event_type}_{e['id']}", "event_type": event_type,
                          "event_name": f"{team} vs {teams['away']}", "venue": match["venue"]["fullName"],
                          "start_utc": e["date"], "status": match["status"]["type"]["name"]})
    return games


sports = pd.DataFrame(
    mlb_home_games("mlb_2025.json") + mlb_home_games("mlb_2026.json")
    + nhl_home_games("nhl_2024_25.json") + nhl_home_games("nhl_2025_26.json")
    + espn_home_games("nba_2025.json", "nba", "Toronto Raptors")
    + espn_home_games("nba_2026.json", "nba", "Toronto Raptors")
    + espn_home_games("nba_2026_playoffs.json", "nba", "Toronto Raptors")
    + espn_home_games("tfc_2025.json", "mls", "Toronto FC")
    + espn_home_games("tfc_2026.json", "mls", "Toronto FC")
)
sports["start"] = (pd.to_datetime(sports.pop("start_utc"), utc=True, format="ISO8601")
                   .dt.tz_convert(TZ).dt.tz_localize(None))
sports["end"] = pd.NaT  # end times are not published
sports["first_day"] = sports["last_day"] = sports["start"].dt.normalize()

# All home games in the files were played (no postponed / cancelled games)
print(sports.groupby(["event_type", "venue"])["status"].value_counts())
sports = sports.drop(columns="status")

event_type  venue             status          
mlb         Rogers Centre     Final               172
mls         BMO Field         STATUS_FULL_TIME     34
nba         Scotiabank Arena  STATUS_FINAL         85
nhl         Scotiabank Arena  OFF                  89
                              FINAL                 6
Name: count, dtype: int64

In [9]:
# Check the UTC → Toronto conversion on the 2025 World Series home games
sports[sports["start"].between("2025-10-24", "2025-11-02")][["event_name", "start"]]

,event_name,start
87,Blue Jays vs Los Angeles Dodgers,2025-10-24 20:00:00
88,Blue Jays vs Los Angeles Dodgers,2025-10-25 20:00:00
89,Blue Jays vs Los Angeles Dodgers,2025-10-31 20:00:00
90,Blue Jays vs Los Angeles Dodgers,2025-11-01 20:00:00
232,Maple Leafs vs BUF,2025-10-25 17:00:00
233,Maple Leafs vs CGY,2025-10-28 18:00:00
308,Toronto Raptors vs Milwaukee Bucks,2025-10-24 18:30:00
309,Toronto Raptors vs Houston Rockets,2025-10-29 18:30:00


### 3.2 Major city events
- From the hand-made `events_manual.csv` (every row has a source link).
- Multi-day events without a start time (CNE, TIFF, marathon) count as event days but get no hourly window.
- `football_match` is renamed `world_cup` (all are FIFA World Cup 2026 matches).

In [10]:
manual = pd.read_csv(RAW / "events_manual.csv")
city_events = pd.DataFrame({
    "event_id": "manual_" + manual.index.astype(str),
    "event_type": manual["category"].replace({"football_match": "world_cup"}),
    "event_name": manual["event_name"],
    "venue": manual["location"],
    "first_day": pd.to_datetime(manual["start_date"]),
    "last_day": pd.to_datetime(manual["end_date"]),
    # A missing time gives NaT (no hourly window)
    "start": pd.to_datetime(manual["start_date"] + " " + manual["start_time"]),
    "end": pd.to_datetime(manual["end_date"] + " " + manual["end_time"]),
})
city_events[["event_type", "event_name", "first_day", "last_day", "start", "end"]]

,event_type,event_name,first_day,last_day,start,end
0,parade,Pride Parade 2025,2025-06-29,2025-06-29,2025-06-29 14:00:00,NaT
1,parade,Toronto Caribbean Carnival Grand Parade 2025,2025-08-02,2025-08-02,2025-08-02 08:00:00,2025-08-02 20:00:00
2,festival,CNE 2025,2025-08-15,2025-09-01,NaT,NaT
3,festival,TIFF 2025,2025-09-04,2025-09-14,NaT,NaT
4,festival,Nuit Blanche 2025,2025-10-04,2025-10-05,2025-10-04 19:00:00,2025-10-05 07:00:00
5,road_race,TCS Toronto Waterfront Marathon 2025,2025-10-18,2025-10-19,NaT,NaT
6,parade,Santa Claus Parade 2025,2025-11-23,2025-11-23,2025-11-23 12:30:00,NaT
7,world_cup,FIFA World Cup: Canada vs Bosnia and Herzegovina,2026-06-12,2026-06-12,2026-06-12 15:00:00,NaT
8,world_cup,FIFA World Cup: Ghana vs Panama,2026-06-17,2026-06-17,2026-06-17 19:00:00,NaT
9,world_cup,FIFA World Cup: Germany vs Ivory Coast,2026-06-20,2026-06-20,2026-06-20 16:00:00,NaT


### 3.3 Time windows and one row per event-day
- Window = 2 h before the start → 2 h after the end (end unknown → start + 3 h).
- Multi-day events are expanded to one row per day; days outside the analysis period are dropped.

In [11]:
events_all = pd.concat([sports, city_events], ignore_index=True)

end_or_default = events_all["end"].fillna(events_all["start"] + pd.Timedelta(hours=DEFAULT_DURATION))
events_all["window_start"] = events_all["start"] - pd.Timedelta(hours=HOURS_BEFORE)
events_all["window_end"] = end_or_default + pd.Timedelta(hours=HOURS_AFTER)

events_all["date"] = [pd.date_range(first, last) for first, last in zip(events_all["first_day"], events_all["last_day"])]
events = (
    events_all.explode("date")
    .assign(date=lambda d: pd.to_datetime(d["date"]))
    .query("@START <= date <= @END")
    [["date", "event_id", "event_type", "event_name", "venue", "start", "end", "window_start", "window_end"]]
    .sort_values(["date", "start"])
    .reset_index(drop=True)
)
events.groupby("event_type").agg(events=("event_id", "nunique"), event_days=("date", "size"))

,events,event_days
event_type,,
festival,5,60
mlb,172,172
mls,31,31
nba,68,68
nhl,70,70
parade,5,5
road_race,1,2
world_cup,6,6


## 4. Weather (Toronto Pearson, daily)
- Keep the variables that can affect bus service.
- 2025-02-25 has no rain/snow values, but total precipitation was only 0.2 mm → set to 0.
- `snow_3d_cm` = snowfall over the last 3 days (snow keeps affecting roads after it falls).

In [12]:
weather = (
    pd.concat([pd.read_csv(RAW / f, encoding="utf-8-sig") for f in ["weather_2025.csv", "weather_2026.csv"]])
    .rename(columns={"Date/Time": "date", "Mean Temp (°C)": "temp_mean_c", "Total Rain (mm)": "rain_mm",
                     "Total Snow (cm)": "snow_cm", "Total Precip (mm)": "precip_mm"})
    .assign(date=lambda d: pd.to_datetime(d["date"]))
    .query("@START <= date <= @END")
    [["date", "temp_mean_c", "rain_mm", "snow_cm", "precip_mm"]]
    .reset_index(drop=True)
)
print("Days with missing values:")
print(weather[weather.isna().any(axis=1)])

weather[["rain_mm", "snow_cm"]] = weather[["rain_mm", "snow_cm"]].fillna(0)
weather["snow_3d_cm"] = weather["snow_cm"].rolling(3, min_periods=1).sum()
assert weather.notna().all().all() and len(weather) == (END - START).days + 1
weather.describe().round(1)

Days with missing values:
         date  temp_mean_c  rain_mm  snow_cm  precip_mm
55 2025-02-25          4.2      NaN      NaN        0.2


,date,temp_mean_c,rain_mm,snow_cm,precip_mm,snow_3d_cm
count,638,638.0,638.0,638.0,638.0,638.0
mean,2025-11-15 12:00:00,9.8,2.0,0.5,2.3,1.4
min,2025-01-01 00:00:00,-16.9,0.0,0.0,0.0,0.0
25%,2025-06-09 06:00:00,0.3,0.0,0.0,0.0,0.0
50%,2025-11-15 12:00:00,10.8,0.0,0.0,0.0,0.0
75%,2026-04-23 18:00:00,20.0,0.3,0.0,1.2,0.2
max,2026-09-30 00:00:00,30.2,48.4,46.2,48.4,47.2
std,NaN,11.3,6.0,2.5,6.2,4.8


## 5. Calendar and holidays
One row per day: weekday, month, weekend and Ontario statutory holidays (package `holidays`).

In [13]:
ontario_holidays = holidays.Canada(subdiv="ON", years=[2025, 2026])

calendar = pd.DataFrame({"date": pd.date_range(START, END)})
calendar["year"] = calendar["date"].dt.year
calendar["month"] = calendar["date"].dt.month
calendar["weekday"] = calendar["date"].dt.day_name()
calendar["is_weekend"] = calendar["date"].dt.day_of_week >= 5
calendar["holiday"] = calendar["date"].dt.date.map(ontario_holidays.get)
calendar["is_holiday"] = calendar["holiday"].notna()
calendar[calendar["is_holiday"]]

,date,year,month,weekday,is_weekend,holiday,is_holiday
0,2025-01-01,2025,1,Wednesday,False,New Year's Day,True
47,2025-02-17,2025,2,Monday,False,Family Day,True
107,2025-04-18,2025,4,Friday,False,Good Friday,True
138,2025-05-19,2025,5,Monday,False,Victoria Day,True
181,2025-07-01,2025,7,Tuesday,False,Canada Day,True
243,2025-09-01,2025,9,Monday,False,Labour Day,True
285,2025-10-13,2025,10,Monday,False,Thanksgiving Day,True
358,2025-12-25,2025,12,Thursday,False,Christmas Day,True
359,2025-12-26,2025,12,Friday,False,Boxing Day,True
365,2026-01-01,2026,1,Thursday,False,New Year's Day,True


## 6. Analysis tables

### 6.1 Day features
Calendar + weather + event flags for every day: `n_events`, `has_event` and one `has_<type>` flag per event type.

In [14]:
event_flags = (pd.crosstab(events["date"], events["event_type"]) > 0).add_prefix("has_").rename_axis(columns=None)
event_flags["n_events"] = events.groupby("date")["event_id"].nunique()

day_features = (
    calendar.merge(weather, on="date", how="left", validate="one_to_one")
    .merge(event_flags.reset_index(), on="date", how="left", validate="one_to_one")
)
flag_cols = [c for c in day_features.columns if c.startswith("has_")]
day_features[flag_cols] = day_features[flag_cols].fillna(False).astype(bool)
day_features["n_events"] = day_features["n_events"].fillna(0).astype(int)
day_features["has_event"] = day_features["n_events"] > 0
print("Days with at least one event:", day_features["has_event"].sum(), "of", len(day_features))

Days with at least one event: 341 of 638


### 6.2 Hourly event windows
Full grid of hours. `n_events_active` = number of events whose time window covers that hour.

In [15]:
windows = events.drop_duplicates("event_id").dropna(subset=["window_start"])
window_hours = pd.Series([
    hour
    for w_start, w_end in zip(windows["window_start"], windows["window_end"])
    for hour in pd.date_range(w_start.floor("h"), w_end, freq="h", inclusive="left")
])

hourly = pd.DataFrame({"datetime": pd.date_range(START, END + pd.Timedelta(hours=23), freq="h")})
hourly["date"] = hourly["datetime"].dt.normalize()
hourly["hour"] = hourly["datetime"].dt.hour
hourly["n_events_active"] = hourly["datetime"].map(window_hours.value_counts()).fillna(0).astype(int)
hourly["in_event_window"] = hourly["n_events_active"] > 0
print("Hours inside an event window:", hourly["in_event_window"].sum(), "of", len(hourly))

Hours inside an event window: 2546 of 15312


### 6.3 Incidents table
Each incident gets its hour's event-window flag and its day's features (many-to-one joins: no row is duplicated).

In [16]:
incidents = (
    ttc.merge(hourly[["date", "hour", "n_events_active", "in_event_window"]],
              on=["date", "hour"], how="left", validate="many_to_one")
    .merge(day_features, on="date", how="left", validate="many_to_one")
)
assert len(incidents) == len(ttc)
incidents.head()

,date,datetime,hour,line,location,code,min_delay,min_gap,route,code_description,...,has_festival,has_mlb,has_mls,has_nba,has_nhl,has_parade,has_road_race,has_world_cup,n_events,has_event
0,2025-01-01,2025-01-01 02:15:00,2,102 MARKHAM ROAD,WARDEN STATION,MFESA,20,40,102,NO OPERATOR AVAILABLE DUE TO ESA,...,False,False,False,True,False,False,False,False,1,True
1,2025-01-01,2025-01-01 02:15:00,2,65 PARLIAMENT,KIPLING STATION,MFUS,0,0,65,UNABLE TO MAINTAIN SCHEDULE,...,False,False,False,True,False,False,False,False,1,True
2,2025-01-01,2025-01-01 02:40:00,2,64 MAIN,BROADVIEW STATION,MFUI,0,0,64,ILL CUSTOMER - TRANSPORTED,...,False,False,False,True,False,False,False,False,1,True
3,2025-01-01,2025-01-01 02:43:00,2,100 FLEMINGDON PARK,OVERLEA AND THORNCLIFF,MFSAN,17,32,100,UNSANITARY,...,False,False,False,True,False,False,False,False,1,True
4,2025-01-01,2025-01-01 03:05:00,3,34 EGLINTON EAST,EGLINTON AND DON MILLS,MFUI,20,40,34,ILL CUSTOMER - TRANSPORTED,...,False,False,False,True,False,False,False,False,1,True


### 6.4 Route distance to venues
`incidents` only knows the route number. The GTFS feed (cleaned by `scripts/clean_gtfs.py`) gives the stops of every route, so we measure
how close each route gets to each venue:
- `dist_<venue>_km` = distance from the venue to the **nearest stop of the route** (straight line, haversine).
- `dist_venue_km` = the smallest of the three; `near_venue` = within `NEAR_KM` of any venue.
- Routes missing from the GTFS feed (e.g. 999, 600, 56, 124, 202; the feed is dated Nov-Dec 2026) and incidents without a route number stay missing (`NaN`), not guessed.
- Venue coordinates are typed in by hand. BMO Field is on the Exhibition Place grounds, so it also covers the CNE.

In [ ]:
import numpy as np

NEAR_KM = 1.0  # a route is "near" a venue if one of its stops is within 1 km
VENUES = {"rogers": (43.6414, -79.3894),      # Rogers Centre
          "scotiabank": (43.6435, -79.3791),  # Scotiabank Arena
          "bmo": (43.6332, -79.4186)}         # BMO Field / Exhibition Place

gtfs_stops = pd.read_csv(PROCESSED / "gtfs_stops.csv", dtype={"stop_id": str})
gtfs_route_stops = pd.read_csv(PROCESSED / "gtfs_route_stops.csv", dtype=str)


def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 2 * 6371 * np.arcsin(np.sqrt(a))


route_stops_xy = gtfs_route_stops.merge(gtfs_stops, on="stop_id", validate="many_to_one")
for venue, (lat, lon) in VENUES.items():
    route_stops_xy[f"dist_{venue}_km"] = haversine_km(route_stops_xy["stop_lat"], route_stops_xy["stop_lon"], lat, lon)

dist_cols = [f"dist_{v}_km" for v in VENUES]
route_venue_dist = route_stops_xy.groupby("route_short_name")[dist_cols].min().round(3)
route_venue_dist["dist_venue_km"] = route_venue_dist.min(axis=1)
route_venue_dist["near_venue"] = route_venue_dist["dist_venue_km"] <= NEAR_KM
# GTFS route names are text; the incident route is a number. A few GTFS routes are not numeric (dropped).
route_venue_dist.index = pd.to_numeric(route_venue_dist.index, errors="coerce")
route_venue_dist = route_venue_dist[route_venue_dist.index.notna()].rename_axis("route").reset_index()
route_venue_dist["route"] = route_venue_dist["route"].astype("Int64")

print("Routes with a distance:", len(route_venue_dist), "| near a venue:", route_venue_dist["near_venue"].sum())
route_venue_dist.sort_values("dist_venue_km").head(10)

In [ ]:
incidents = incidents.merge(route_venue_dist, on="route", how="left", validate="many_to_one")
assert len(incidents) == len(ttc)

no_geo = incidents["dist_venue_km"].isna()
print(f"Incidents without route distance: {no_geo.sum():,} ({no_geo.mean():.1%})")
print("Routes not in GTFS (top):", incidents.loc[no_geo & incidents['route'].notna(), 'route'].value_counts().head(8).to_dict())
print(f"Incidents on a route near a venue: {incidents['near_venue'].sum():,}")

### 6.5 Hourly and daily tables
Counts per hour / per day. Hours with no incidents stay in the table with 0 (otherwise averages would be too high).

In [17]:
def count_incidents(df, keys):
    """Incident counts and delay minutes per group (e.g. per day or per hour)."""
    with_flags = df.assign(**{col: df["code"].eq(code) for code, col in KEY_CODES.items()})
    return with_flags.groupby(keys).agg(
        n_incidents=("code", "size"),
        n_delays=("is_delay", "sum"),
        delay_min_total=("delay_capped", "sum"),
        **{col: (col, "sum") for col in KEY_CODES.values()},
    ).reset_index()


count_cols = ["n_incidents", "n_delays", "delay_min_total", *KEY_CODES.values()]

hourly = (
    hourly.merge(count_incidents(ttc, ["date", "hour"]), on=["date", "hour"], how="left", validate="one_to_one")
    .merge(day_features, on="date", how="left", validate="many_to_one")
)
hourly[count_cols] = hourly[count_cols].fillna(0).astype(int)

median_delay = ttc[ttc["is_delay"]].groupby("date")["min_delay"].median().rename("delay_min_median").reset_index()
daily = (
    day_features.merge(count_incidents(ttc, ["date"]), on="date", how="left", validate="one_to_one")
    .merge(median_delay, on="date", how="left", validate="one_to_one")
)
daily.head()

,date,year,month,weekday,is_weekend,holiday,is_holiday,temp_mean_c,rain_mm,snow_cm,...,has_world_cup,n_events,has_event,n_incidents,n_delays,delay_min_total,n_diversion,n_parade_hold,n_behind_schedule,delay_min_median
0,2025-01-01,2025,1,Wednesday,False,New Year's Day,True,1.3,0.7,0.0,...,False,1,True,80,70,1868,8,0,3,16.5
1,2025-01-02,2025,1,Thursday,False,NaN,False,-1.5,0.0,1.8,...,False,0,False,118,102,1663,9,0,1,10.0
2,2025-01-03,2025,1,Friday,False,NaN,False,-2.7,0.0,0.0,...,False,1,True,118,103,2045,7,0,0,12.0
3,2025-01-04,2025,1,Saturday,True,NaN,False,-4.4,0.0,0.0,...,False,1,True,113,103,2307,2,1,1,15.0
4,2025-01-05,2025,1,Sunday,True,NaN,False,-3.6,0.0,0.0,...,False,1,True,68,59,1253,4,0,0,15.0


## 7. Checks
Totals must agree across tables, keys must be unique, no missing values in key columns.

In [18]:
assert hourly["n_incidents"].sum() == daily["n_incidents"].sum() == len(incidents) == len(ttc)
assert len(daily) == len(calendar) and not daily["date"].duplicated().any()
assert len(hourly) == len(calendar) * 24 and not hourly.duplicated(["date", "hour"]).any()
assert daily[count_cols + ["delay_min_median"]].notna().all().all()
assert events_all["event_id"].is_unique
print("All checks passed")

All checks passed


## 8. Save

In [19]:
PROCESSED.mkdir(exist_ok=True)
tables = {"incidents": incidents, "events": events, "daily": daily, "hourly": hourly,
          "route_venue_dist": route_venue_dist}
for name, table in tables.items():
    table.to_csv(PROCESSED / f"{name}.csv", index=False)

pd.DataFrame({name: {"rows": len(t), "columns": t.shape[1]} for name, t in tables.items()}).T

,rows,columns
incidents,107049,36
events,414,9
daily,638,29
hourly,15312,32


## Constructed variables

| Column | Tables | Meaning |
|---|---|---|
| `route` | incidents | route number taken from `line` |
| `dist_rogers_km`, `dist_scotiabank_km`, `dist_bmo_km` | incidents, route_venue_dist | km from the venue to the nearest stop of the route (GTFS stops, straight line) |
| `dist_venue_km`, `near_venue` | incidents, route_venue_dist | closest of the three venues / within `NEAR_KM` (1 km); missing if the route is not in the GTFS feed |
| `cause_group` | incidents | code family by first letter (Equipment, Miscellaneous, Security, Operator, Other) |
| `is_delay` | incidents | `min_delay > 0` |
| `delay_capped` | incidents | `min_delay` capped at the 99th percentile |
| `in_event_window`, `n_events_active` | incidents, hourly | hour is inside ≥ 1 event window / number of such events |
| `has_event`, `n_events`, `has_<type>` | all except events | day has an event / number of events / event of this type |
| `snow_3d_cm` | daily, hourly, incidents | snowfall over the last 3 days |
| `n_incidents`, `n_delays` | daily, hourly | all incidents / incidents with delay > 0 |
| `delay_min_total` | daily, hourly | sum of `delay_capped` |
| `delay_min_median` | daily | median delay of incidents with delay > 0 |
| `n_diversion`, `n_parade_hold`, `n_behind_schedule` | daily, hourly | incidents with codes MFDV, MFPR, MFUS |